# 遞迴密技：Cache / LRU Cache / 動態規劃 DP

1. 為什麼普通遞迴 Fibonacci 很慢
2. `functools.cache` / `functools.lru_cache` 怎麼加速
3. 什麼是「記憶化搜尋」Memoization
4. 用 AtCoder DP H - Grid 1 示範遞迴 + cache
5. 再改寫成 Bottom-up DP

核心觀念：

> 遞迴慢，常常不是因為「遞迴」本身，而是因為「同一個狀態一直被重複計算」。

## 0. 測速工具

`perf_counter()` 適合用來測比較短的程式執行時間。

In [1]:
from time import perf_counter


def show_time(title, func, *args):
    start = perf_counter()
    result = func(*args)
    end = perf_counter()
    print(title)
    print('Result:', result)
    print(f'Time: {end - start:.6f} seconds')
    print('-' * 40)
    return result

## 1. 普通遞迴 Fibonacci

Fibonacci 定義：

```text
fib(0) = 0
fib(1) = 1
fib(n) = fib(n - 1) + fib(n - 2)
```

直接照定義寫很直覺，但會非常慢。

In [2]:
def fib_plain(n):
    if n < 2:
        return n
    return fib_plain(n - 1) + fib_plain(n - 2)

show_time('普通遞迴 fib_plain(30)', fib_plain, 30)

普通遞迴 fib_plain(30)
Result: 832040
Time: 0.097677 seconds
----------------------------------------


832040

### 為什麼 Fibonacci 遞迴會慢？

例如：

```text
fib(5)
= fib(4) + fib(3)
= fib(3) + fib(2) + fib(2) + fib(1)
```

你會發現 `fib(3)`、`fib(2)` 會被重複算很多次。

所以問題是：

> 同一個子問題重複計算。

## 2. 加上 `@cache`

`@cache` 會幫你把函式結果記起來。

概念上像這樣：

```text
fib_cache(10) 算完後，Python 幫你記住：10 -> 55
下次再問 fib_cache(10)，直接回傳 55
```

In [3]:
from functools import cache

@cache
def fib_cache(n):
    if n < 2:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)

show_time('加上 @cache 的 fib_cache(80)', fib_cache, 80)
print(fib_cache.cache_info())

加上 @cache 的 fib_cache(80)
Result: 23416728348467685
Time: 0.000077 seconds
----------------------------------------
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


`cache_info()` 可以看到：

- `hits`：從快取拿答案的次數
- `misses`：第一次真的計算的次數
- `currsize`：目前快取中有幾筆答案

## 3. `@lru_cache(maxsize=None)`

舊一點或比賽常見寫法是：

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def f(...):
    ...
```

`maxsize=None` 表示不限制快取大小。

在這種 DP 題中，它的用途和 `@cache` 很像。

In [4]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_lru(n):
    if n < 2:
        return n
    return fib_lru(n - 1) + fib_lru(n - 2)

show_time('加上 @lru_cache(None) 的 fib_lru(80)', fib_lru, 80)
print(fib_lru.cache_info())

加上 @lru_cache(None) 的 fib_lru(80)
Result: 23416728348467685
Time: 0.000036 seconds
----------------------------------------
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


## 4. 手刻 memo

`@cache` / `@lru_cache` 的概念，其實就是用字典存答案。

In [5]:
def fib_memo(n, memo=None):
    if memo is None:
        memo = {}

    if n in memo:
        return memo[n]

    if n < 2:
        memo[n] = n
    else:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)

    return memo[n]

show_time('手刻 memo 的 fib_memo(80)', fib_memo, 80)

手刻 memo 的 fib_memo(80)
Result: 23416728348467685
Time: 0.000030 seconds
----------------------------------------


23416728348467685

## 5. cache 的參數要可以 hash

因為 cache 內部會把函式參數當成字典 key。

通常可以：`int`、`str`、`tuple`

通常不行：`list`、`dict`

In [6]:
from functools import cache

@cache
def bad_example(x):
    return sum(x)

try:
    bad_example([1, 2, 3])
except TypeError as e:
    print('錯誤訊息:', e)

print('改成 tuple 就可以：', bad_example((1, 2, 3)))

錯誤訊息: unhashable type: 'list'
改成 tuple 就可以： 6


---

# 6. AtCoder DP H - Grid 1

題目：給定 `H × W` 的網格。

- `.` 表示空格
- `#` 表示障礙物
- 從左上角走到右下角
- 每次只能向右或向下
- 求不同路徑數
- 答案要取 `10^9 + 7` 的餘數

範例輸入：

```text
3 4
...#
.#..
....
```

範例輸出：

```text
3
```

## 7. 用字串模擬輸入

Notebook 教學先用字串模擬輸入，比較方便觀察。

In [7]:
sample_input = '3 4\n...#\n.#..\n....\n'

print(sample_input)

3 4
...#
.#..
....



In [8]:
def parse_grid(text):
    lines = text.strip().splitlines()
    H, W = map(int, lines[0].split())
    grid = lines[1:]
    return H, W, grid

H, W, grid = parse_grid(sample_input)
print(H, W)
for row in grid:
    print(row)

3 4
...#
.#..
....


## 8. 普通遞迴版本

定義：

```text
count_paths(i, j) = 從 (i, j) 走到終點的路徑數
```

遞迴關係：

```text
count_paths(i, j) = count_paths(i + 1, j) + count_paths(i, j + 1)
```

也就是：

```text
往下走的路徑數 + 往右走的路徑數
```

In [9]:
MOD = 10**9 + 7

def solve_recursive_plain(text):
    H, W, grid = parse_grid(text)

    def count_paths(i, j):
        # 越界，沒有路
        if i >= H or j >= W:
            return 0

        # 遇到障礙物，沒有路
        if grid[i][j] == '#':
            return 0

        # 到達終點，有 1 條路
        if i == H - 1 and j == W - 1:
            return 1

        return (count_paths(i + 1, j) + count_paths(i, j + 1)) % MOD

    return count_paths(0, 0)

show_time('普通遞迴解範例', solve_recursive_plain, sample_input)

普通遞迴解範例
Result: 3
Time: 0.000018 seconds
----------------------------------------


3

普通遞迴在小範例可以跑。

但網格變大時，很多格子的答案會一直重算。

例如 `(1, 1)` 可能從上面走到，也可能從左邊走到。沒有 cache 時，後面路徑會被重複展開。

## 9. 加上 `@lru_cache`：Top-down DP / 記憶化搜尋

把每一格 `(i, j)` 的答案記起來。

同一格第二次被問到時，直接回傳答案。

In [10]:
from functools import lru_cache


def solve_recursive_cache(text, verbose=False):
    H, W, grid = parse_grid(text)

    @lru_cache(maxsize=None)
    def count_paths(i, j):
        if verbose:
            print('真的計算:', i, j)

        if i >= H or j >= W:
            return 0
        if grid[i][j] == '#':
            return 0
        if i == H - 1 and j == W - 1:
            return 1

        return (count_paths(i + 1, j) + count_paths(i, j + 1)) % MOD

    ans = count_paths(0, 0)
    return ans, count_paths.cache_info()

ans, info = solve_recursive_cache(sample_input, verbose=True)
print('答案:', ans)
print(info)

真的計算: 0 0
真的計算: 1 0
真的計算: 2 0
真的計算: 3 0
真的計算: 2 1
真的計算: 3 1
真的計算: 2 2
真的計算: 3 2
真的計算: 2 3
真的計算: 1 1
真的計算: 0 1
真的計算: 0 2
真的計算: 1 2
真的計算: 1 3
真的計算: 1 4
真的計算: 0 3
答案: 3
CacheInfo(hits=3, misses=16, maxsize=None, currsize=16)


`verbose=True` 會印出真正計算的狀態。

如果某個 `(i, j)` 已經算過，第二次會直接從 cache 拿答案，不會再印出來。

## 10. 比較普通遞迴 vs cache 遞迴

下面建立一個沒有障礙物的 `12 × 12` 網格。

普通遞迴會分裂出大量重複子問題；cache 版本每一格最多算一次。

In [11]:
def make_empty_grid_input(H, W):
    rows = ['.' * W for _ in range(H)]
    return f'{H} {W}\n' + '\n'.join(rows)

small_grid = make_empty_grid_input(12, 12)

show_time('12x12 普通遞迴', solve_recursive_plain, small_grid)
ans, info = show_time('12x12 cache 遞迴', solve_recursive_cache, small_grid)
print(info)

12x12 普通遞迴
Result: 705432
Time: 0.528942 seconds
----------------------------------------
12x12 cache 遞迴
Result: (705432, CacheInfo(hits=121, misses=166, maxsize=None, currsize=166))
Time: 0.000101 seconds
----------------------------------------
CacheInfo(hits=121, misses=166, maxsize=None, currsize=166)


重點：

```text
普通遞迴：接近指數級，因為一直重算
加 cache：O(H × W)，每一格最多算一次
```

## 11. 比賽提交版：Top-down DP

這版使用：

- `sys.stdin.read()` 讀輸入
- `@lru_cache(maxsize=None)` 記憶化搜尋
- `sys.setrecursionlimit()` 避免遞迴太深

In [12]:
%%writefile grid_top_down.py
import sys
from functools import lru_cache

sys.setrecursionlimit(1 << 25)
MOD = 10**9 + 7

def main():
    data = sys.stdin.read().splitlines()
    H, W = map(int, data[0].split())
    grid = data[1:]

    @lru_cache(maxsize=None)
    def count_paths(i, j):
        if i >= H or j >= W:
            return 0
        if grid[i][j] == '#':
            return 0
        if i == H - 1 and j == W - 1:
            return 1
        return (count_paths(i + 1, j) + count_paths(i, j + 1)) % MOD

    print(count_paths(0, 0))

if __name__ == '__main__':
    main()

Writing grid_top_down.py


In [13]:
%%writefile sample_grid.txt
3 4
...#
.#..
....

Writing sample_grid.txt


In [14]:
!python grid_top_down.py < sample_grid.txt

3


## 12. Bottom-up DP：不用遞迴的版本

定義：

```text
dp[i][j] = 從左上角走到 (i, j) 的路徑數
```

轉移：

```text
dp[i][j] += dp[i - 1][j]  # 從上面來
dp[i][j] += dp[i][j - 1]  # 從左邊來
```

In [15]:
def solve_bottom_up(text):
    H, W, grid = parse_grid(text)
    dp = [[0] * W for _ in range(H)]

    for i in range(H):
        for j in range(W):
            if grid[i][j] == '#':
                continue

            if i == 0 and j == 0:
                dp[i][j] = 1
            else:
                if i > 0:
                    dp[i][j] += dp[i - 1][j]
                if j > 0:
                    dp[i][j] += dp[i][j - 1]
                dp[i][j] %= MOD

    return dp[H - 1][W - 1], dp

ans, dp = solve_bottom_up(sample_input)
print('答案:', ans)
print('DP 表：')
for row in dp:
    print(row)

答案: 3
DP 表：
[1, 1, 1, 0]
[1, 0, 1, 1]
[1, 1, 2, 3]


這個表可以這樣解讀：

```text
dp[i][j] = 走到這格有幾種方法
```

障礙物那格會保持 0。

## 13. 比賽提交版：Bottom-up DP

這版不使用遞迴，比較不會遇到 recursion limit 問題。

In [16]:
%%writefile grid_bottom_up.py
import sys

MOD = 10**9 + 7

def main():
    data = sys.stdin.read().splitlines()
    H, W = map(int, data[0].split())
    grid = data[1:]

    dp = [[0] * W for _ in range(H)]

    for i in range(H):
        for j in range(W):
            if grid[i][j] == '#':
                continue

            if i == 0 and j == 0:
                dp[i][j] = 1
            else:
                if i > 0:
                    dp[i][j] += dp[i - 1][j]
                if j > 0:
                    dp[i][j] += dp[i][j - 1]
                dp[i][j] %= MOD

    print(dp[H - 1][W - 1])

if __name__ == '__main__':
    main()

Writing grid_bottom_up.py


In [17]:
!python grid_bottom_up.py < sample_grid.txt

3


## 14. Top-down vs Bottom-up

| 寫法 | 優點 | 缺點 |
|---|---|---|
| Top-down 遞迴 + cache | 接近人類思考，從目標問題往下拆 | 可能遇到遞迴深度問題 |
| Bottom-up DP | 穩定、不怕遞迴深度 | 需要先想好填表順序 |

這題兩種方法都是：

```text
時間複雜度：O(H × W)
空間複雜度：O(H × W)
```

## 15. 什麼時候適合用 cache？

適合：

- 有重複子問題
- 狀態可以用參數表示
- 同一個狀態會被問很多次

常見例子：

```text
Fibonacci
格子路徑
背包問題
字串 DP
區間 DP
遊戲狀態搜尋
```

不適合：

- 每次輸入都不同，幾乎不會重複
- 參數是 list / dict 這種不能 hash 的物件
- 狀態數太多，記憶體會爆

## 16. 模板

### 遞迴 + cache 模板

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def dp(state):
    if 終止條件:
        return 答案

    return 子問題答案合併
```

### Grid 題核心模板

```python
@lru_cache(maxsize=None)
def count_paths(i, j):
    if i >= H or j >= W or grid[i][j] == '#':
        return 0
    if i == H - 1 and j == W - 1:
        return 1
    return (count_paths(i + 1, j) + count_paths(i, j + 1)) % MOD
```

## 17. 課堂練習

### 練習 1

把 Fibonacci 改成：

```text
tribonacci(n) = tribonacci(n-1) + tribonacci(n-2) + tribonacci(n-3)
```

其中：

```text
tribonacci(0) = 0
tribonacci(1) = 1
tribonacci(2) = 1
```

請用 `@lru_cache(None)` 寫。

### 練習 2

修改 Grid 題：

原本只能往右、往下。

現在可以：

```text
往右、往下、右下斜走
```

請修改 `count_paths(i, j)`。

### 練習 3

為什麼 `@lru_cache` 裡面的函式不要依賴會改變的全域變數？

## 18. 參考答案

In [18]:
from functools import lru_cache

@lru_cache(maxsize=None)
def tribonacci(n):
    if n == 0:
        return 0
    if n == 1 or n == 2:
        return 1
    return tribonacci(n - 1) + tribonacci(n - 2) + tribonacci(n - 3)

print([tribonacci(i) for i in range(10)])
print('tribonacci(30) =', tribonacci(30))

[0, 1, 1, 2, 4, 7, 13, 24, 44, 81]
tribonacci(30) = 29249425


In [19]:
def solve_grid_with_diagonal(text):
    H, W, grid = parse_grid(text)

    @lru_cache(maxsize=None)
    def count_paths(i, j):
        if i >= H or j >= W:
            return 0
        if grid[i][j] == '#':
            return 0
        if i == H - 1 and j == W - 1:
            return 1

        return (
            count_paths(i + 1, j) +      # 往下
            count_paths(i, j + 1) +      # 往右
            count_paths(i + 1, j + 1)    # 右下斜走
        ) % MOD

    return count_paths(0, 0)

print(solve_grid_with_diagonal(sample_input))

9


練習 3 參考說法：

`@lru_cache` 只看函式參數是否一樣。

如果函式內部使用了會改變的全域變數，但是參數沒變，cache 仍然會回傳舊答案。

所以 DP 函式最好讓「會影響答案的東西」都固定，或確定它在計算過程中不會改變。